# Индивидуальный проект — начало...

## Данные + первый индекс

Цель пары:

1. выбрать датасет, который подойдёт для дальнейшего проекта
2. проверить, что в нём есть нужные типы полей
3. выбрать **одно уникальное поле**
4. реализовать свою `Hash Table`
5. добавить `resize` и `rehash`
6. использовать таблицу как индекс вида `уникальный ключ → row_id`
7. сравнить поиск полным проходом и поиск через индекс

На следующих занятиях проект будет постепенно расширяться новыми структурами и алгоритмами, поэтому к выбору датасета нужно подойти ответственно


---
# 1. Требования к датасету

Выберите датасет, с которым будете работать дальше в течение этого семестра

### Обязательные требования

Датасет должен содержать:

- желательно **не менее 100 000 строк**
- минимум **5 содержательных столбцов**
- минимум **одно уникальное поле** — идентификатор записи
- минимум **одно числовое поле**
- минимум **одно категориальное поле**
- **два поля, которые описывают связь между объектами**, и ещё одно числовое поле, которое характеризует эту связь. Например:
  - аэропорт отправления → аэропорт назначения, число — длительность полёта / цена билета / расстояние
  - отправитель → получатель, число — сумма перевода
  - пользователь → товар, число — цена / количество / оценка
  - пользователь → трек, число — количество прослушиваний / оценка / длительность прослушивания
  - команда 1 → команда 2, число — разница счёта / количество очков / длительность матча
  - пользователь 1 → пользователь 2, число — количество сообщений / взаимодействий / совместных действий
- минимум **одно поле, по которому значения можно упорядочить**:
  - число
  - дата/время
  - рейтинг
  - длительность и т. п.

### Желательно

Хорошо, если есть дата/время — это пригодится в следующих задачах

### Пример подходящего датасета

Датасет покупок в маркетплейсе:

```text
purchase_id   — уникальный идентификатор покупки
user_id       — идентификатор пользователя
product_id    — идентификатор товара
quantity      — количество купленного товара
price         — цена товара
category      — категория товара
created_at    — дата и время покупки
```

Почему подходит:

- `purchase_id` — уникальное поле
- `price`, `quantity` — числовые поля
- `category` — категориальное поле
- `created_at` — поле, значения которого можно упорядочить
- `user_id → product_id` — связь между двумя объектами: пользователь купил товар
- `quantity` или `price` — числовая характеристика этой связи

Одну строку можно прочитать так:

> Пользователь `user_id` купил товар `product_id` в количестве `quantity` по цене `price` в момент `created_at`.

**Сейчас использовать связь между объектами в коде не нужно. Она понадобится на следующих занятиях**


## Коротко опишите свой датасет

**Название / источник:**  
`https://excelbianalytics.com/downloads-18-sample-csv-files-data-sets-for-testing-sales/$0
`

**Что представляет одна строка:**  
`Информация об одном заказе.`

**Размер датасета:**  
`100 000 строк, 14 столбцов.`

**Какие поля есть:**  
`Region, Country, Item Type, Sales Channel, Order Priority, Order Date, Order ID, Ship Date, Units Sold, Unit Price, Unit Cost, Total Revenue, Total Cost, Total Profit`

**Какое поле является уникальным идентификатором:**  
`Order ID`

**Какое поле числовое:**  
`Total Profit`

**Какое поле категориальное:**  
`Item Type`

**Какое поле можно использовать как упорядочиваемое:**  
`Order Date`

**Какие два поля описывают связь между объектами:**  
`Total Revenue → Total Cost`

**Какое числовое поле характеризует эту связь:**  
`Total Profit - прибыль, рассчитываемая как разница между выручкой и затратами.`

---
# 2. Загружаем данные

## Вариант A — загрузка файла с компа

In [2]:
import pandas as pd
import numpy as np


df = pd.read_csv("sample_data/100000 Sales Records.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Region          100000 non-null  object 
 1   Country         100000 non-null  object 
 2   Item Type       100000 non-null  object 
 3   Sales Channel   100000 non-null  object 
 4   Order Priority  100000 non-null  object 
 5   Order Date      100000 non-null  object 
 6   Order ID        100000 non-null  int64  
 7   Ship Date       100000 non-null  object 
 8   Units Sold      100000 non-null  int64  
 9   Unit Price      100000 non-null  float64
 10  Unit Cost       100000 non-null  float64
 11  Total Revenue   100000 non-null  float64
 12  Total Cost      100000 non-null  float64
 13  Total Profit    100000 non-null  float64
dtypes: float64(5), int64(2), object(7)
memory usage: 10.7+ MB


---
# 3. Выбираем уникальное поле

Сегодня строим индекс только для **уникального ключа**

Примеры:

```text
идентификатор покупки
идентификатор транзакции
идентификатор поездки
идентификатор рейса
```

Для такого поля будем хранить:

```text
уникальный ключ → Order_ID
```

Например:

```text
Order ID 837291 → строка 41827
```


In [3]:
UNIQUE_FIELD = "Order ID"

assert UNIQUE_FIELD in df.columns, (
    f"Столбца {UNIQUE_FIELD!r} нет. "
    f"Доступные столбцы: {list(df.columns)}"
)

series = df[UNIQUE_FIELD]

print("Поле:", UNIQUE_FIELD)
print("Количество строк:", len(series))
print("Уникальных значений:", series.nunique(dropna=False))
print("Пропущенных значений:", series.isna().sum())
print("Поле уникально:", series.is_unique)

Поле: Order ID
Количество строк: 100000
Уникальных значений: 100000
Пропущенных значений: 0
Поле уникально: True


### Требование

Для основной части работы поле должно быть:

- **уникальным**
- **без пропущенных значений**

Если поле не проходит проверку, выберите другое

Если в датасете нет естественного уникального идентификатора — обсудите это с преподавателем


In [4]:
assert series.is_unique, (
    "Выбранное поле не уникально"
    "Для сегодняшнего задания выберите уникальный идентификатор"
)

assert not series.isna().any(), (
    "В уникальном поле есть пропущенные значения"
    "Выберите поле без пропусков"
)

keys = series.tolist()
print("Уникальный ключ выбран")


Уникальный ключ выбран


---
# 4. Поиск без индекса

Сначала реализуем обычный поиск полным проходом

Функция должна вернуть `Order_ID` найденной записи или `None`

In [5]:
keys = df["Order ID"].to_numpy()

def linear_search(values, target):
    for i in range(len(values)):
        if values[i] == target:
            return i
    return None

In [7]:
TARGET = keys[len(keys) // 2]

Order_ID = linear_search(keys, TARGET)

print("TARGET:", TARGET)
print("Order_ID:", Order_ID)

if Order_ID is not None:
    display(df.iloc[[Order_ID]])

TARGET: 238846909
Order_ID: 50000


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
50000,Asia,Myanmar,Beverages,Offline,L,5/29/2016,238846909,6/15/2016,8963,47.45,31.79,425294.35,284933.77,140360.58


### Ответьте

**Временная сложность в худшем случае:**  
O(n)

**Дополнительная память:**  
O(1)

**Почему при миллионах записей такой поиск может быть проблемой:**  
Из-за линейной сложности $O(N)$ компьютеру приходится проверять каждую строку по очереди, и при миллионах записей поиск будет занимать слишком много времени.

---
# 5. Реализуем Hash Table

Идея:

```text
key
 ↓
hash(key)
 ↓
hash(key) % capacity
 ↓
bucket
```

Разные ключи могут попасть в один `bucket` — это **коллизия**

Коллизии в этой работе решаем методом **separate chaining**:

```text
bucket → [(key1, value1), (key2, value2), ...]
```

## Обязательное расширение таблицы

Таблица начинается с небольшой вместимости:

```text
capacity = 8
```

Коэффициент заполнения:

```text
load_factor = size / capacity
```

В нашей реализации используем порог:

```text
MAX_LOAD_FACTOR = 0.75
```

Если добавление **нового** ключа приведёт к:

```text
(size + 1) / capacity > 0.75
```

таблица должна:

1. увеличить `capacity` в 2 раза
2. создать новые пустые `buckets`
3. заново распределить по ним **все старые пары**
4. после этого добавить новый элемент

Это называется **rehash**

Почему нельзя просто увеличить список `buckets`?

Потому что номер bucket вычисляется так:

```text
hash(key) % capacity
```

После изменения `capacity` тот же ключ может получить уже другой номер bucket

> `0.75` — порог, который мы используем в нашей учебной реализации. Это не универсальное правило для всех Hash Table

### Ограничение

Внутри `MyHashMap` нельзя использовать:

- `dict`
- `set`
- `defaultdict`


In [42]:
class HashMap:
    """Функция реализации Хэш Таблицы."""

    MAX_LOAD_FACTOR = 0.75

    def __init__(self, capacity: int = 10) -> None:
        if capacity <= 0:
            raise ValueError("capacity должна быть больше 0")

        self.capacity = capacity
        self.size = 0
        self.buckets = [[] for _ in range(capacity)]

    def _index(self, key) -> int:
        """Вычисляет индекс элемента в Хэш Таблице."""

        hash_value = hash(key)

        hash_value = hash_value % self.capacity

        return hash_value

    def load_factor(self) -> float:
        """Возвращает коэффициент заполнения Хэш Таблицы."""

        return self.size / self.capacity

    def contains(self, key) -> bool:
        """Возвращает True, если элемент есть в Хэш Таблице, иначе False."""

        hash_value = self._index(key)

        hash_map = self.buckets
        bucket = hash_map[hash_value]

        for element in bucket:
            if element[0] == key:
                return True

        return False

    def put(self, key, value) -> None:
        """Кладет элемент в Хэш Таблицу."""

        hash_value = self._index(key)

        hash_map = self.buckets
        bucket = hash_map[hash_value]

        for i, element in enumerate(bucket):
            if element[0] == key:
                bucket[i] = (key, value)
                return

        bucket.append((key, value))
        self.size += 1

        self._resize()

    def get(self, key):
        """Возвращает значение по ключу."""

        hash_value = self._index(key)

        hash_map = self.buckets
        bucket = hash_map[hash_value]

        for element in bucket:
            if element[0] == key:
                return element[1]

        return None

    def remove(self, key) -> bool:
        """Удаляет элемент. Возвращает True, если удалили, иначе False."""

        hash_value = self._index(key)

        hash_map = self.buckets
        bucket = hash_map[hash_value]

        for element in bucket:
            if element[0] == key:
                bucket.remove(element)
                self.size -= 1
                return True

        return False

    def _resize(self) -> None:
        """Увеличивает размер Хэш Таблицы при переполнении элементов."""

        if self.load_factor() > self.MAX_LOAD_FACTOR:
            hash_map = self.buckets

            self.capacity *= 2

            new_buckets = [[] for _ in range(self.capacity)]

            for bucket in hash_map:
                for element in bucket:
                    hash_value = hash(element[0]) % self.capacity
                    new_buckets[hash_value].append(element)

            self.buckets = new_buckets

    def __len__(self) -> int:
        return self.size

---
# 6. Проверяем реализацию

In [12]:
def check(name, condition):
    print(("СУПЕР!!!!" if condition else "МЛДАААА"), name)

m = HashMap(4)

m.put("a", 10)
check("put/get", m.get("a") == 10)

m.put("a", 20)
check("обновление существующего ключа", m.get("a") == 20)
check("size после update не вырос", len(m) == 1)

check("contains", m.contains("a"))
check("contains missing", not m.contains("missing"))

check("remove", m.remove("a") is True)
check("ключ удалён", not m.contains("a"))
check("повторный remove", m.remove("a") is False)

# Проверяем, что обновление существующего ключа
# не должно само по себе вызывать resize
m = HashMap(4)
m.put("a", 1)
m.put("b", 2)
m.put("c", 3)          # load factor = 0.75

capacity_before = m.capacity
m.put("a", 100)        # update, а не новый ключ

check("update не вызывает resize", m.capacity == capacity_before)
check("значение после update изменилось", m.get("a") == 100)


СУПЕР!!!! put/get
СУПЕР!!!! обновление существующего ключа
СУПЕР!!!! size после update не вырос
СУПЕР!!!! contains
СУПЕР!!!! contains missing
СУПЕР!!!! remove
СУПЕР!!!! ключ удалён
СУПЕР!!!! повторный remove
СУПЕР!!!! update не вызывает resize
СУПЕР!!!! значение после update изменилось


## Проверка коллизий

Чтобы `resize` не мешал эксперименту, возьмём `capacity = 8`

Ключи:

```text
1, 9, 17, 25
```

при `capacity = 8` имеют одинаковый остаток при делении на 8 и попадают в один bucket

Все значения всё равно должны корректно находиться


In [14]:
m = HashMap(8)

for key in [1, 9, 17, 25]:
    m.put(key, key * 10)

check("collision 1", m.get(1) == 10)
check("collision 9", m.get(9) == 90)
check("collision 17", m.get(17) == 170)
check("collision 25", m.get(25) == 250)
check("resize ещё не нужен", m.capacity == 8)

m.remove(9)

check("9 удалён", m.get(9) is None)
check("1 остался", m.get(1) == 10)
check("17 остался", m.get(17) == 170)
check("25 остался", m.get(25) == 250)


СУПЕР!!!! collision 1
СУПЕР!!!! collision 9
СУПЕР!!!! collision 17
СУПЕР!!!! collision 25
СУПЕР!!!! resize ещё не нужен
СУПЕР!!!! 9 удалён
СУПЕР!!!! 1 остался
СУПЕР!!!! 17 остался
СУПЕР!!!! 25 остался


## Проверка `resize` и `rehash`

При `capacity = 4`:

```text
3 / 4 = 0.75
```

Три элемента еще помещаются без расширения

Но добавление четвертого дало бы:

```text
4 / 4 = 1.0
```

Поэтому **перед добавлением четвёртого нового ключа** таблица должна расшириться:

```text
4 → 8
```

После `resize` все старые элементы должны по-прежнему находиться


In [22]:
m = HashMap(4)

m.put(1, "A")
m.put(2, "B")
m.put(3, "C")

check("после 3 элементов capacity = 4", m.capacity == 4)
check("load factor = 0.75", abs(m.load_factor() - 0.75) < 1e-9)

m.put(4, "D")

check("при добавлении 4-го элемента capacity стала 8", m.capacity == 8)
check("1 не потерялся после rehash", m.get(1) == "A")
check("2 не потерялся после rehash", m.get(2) == "B")
check("3 не потерялся после rehash", m.get(3) == "C")
check("4 добавился", m.get(4) == "D")
check("load factor снова <= 0.75", m.load_factor() <= HashMap.MAX_LOAD_FACTOR)


МЛДАААА после 3 элементов capacity = 4
МЛДАААА load factor = 0.75
СУПЕР!!!! при добавлении 4-го элемента capacity стала 8
СУПЕР!!!! 1 не потерялся после rehash
СУПЕР!!!! 2 не потерялся после rehash
СУПЕР!!!! 3 не потерялся после rehash
СУПЕР!!!! 4 добавился
СУПЕР!!!! load factor снова <= 0.75


---
# 7. Строим индекс на своем датасете

Теперь используем собственную Hash Table как реальный индекс:

```text
уникальный ключ → Order_ID
```

In [43]:
import pandas as pd

df = pd.read_csv("sample_data/100000 Sales Records.csv")

keys = df["Order ID"].to_numpy()

def build_hash_index(values):
    index = HashMap()

    for row_id, key in enumerate(values):
        index.put(key, row_id)

    return index

In [44]:
hash_index = build_hash_index(keys)

print("Индекс построен")
print("Количество ключей:", len(hash_index))
print("Итоговая capacity:", hash_index.capacity)
print(f"Load factor: {hash_index.load_factor():.4f}")

Индекс построен
Количество ключей: 100000
Итоговая capacity: 163840
Load factor: 0.6104


In [45]:
TTARGET = keys[len(keys) // 2]

row_id = hash_index.get(TARGET)

print("TARGET:", TARGET)
print("Order_ID:", row_id)

if row_id is not None:
    display(df.iloc[[row_id]])

TARGET: 238846909
Order_ID: 50000


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
50000,Asia,Myanmar,Beverages,Offline,L,5/29/2016,238846909,6/15/2016,8963,47.45,31.79,425294.35,284933.77,140360.58


---
# 8. Сравниваем два способа поиска

In [47]:
test_positions = np.linspace(
    0,
    len(keys) - 1,
    num=min(30, len(keys)),
    dtype=int
)

test_keys = [keys[i] for i in test_positions]

start = perf_counter()
for key in test_keys:
    linear_search(keys, key)
linear_ms = (perf_counter() - start) * 1000

start = perf_counter()
for key in test_keys:
    hash_index.get(key)
hash_ms = (perf_counter() - start) * 1000

print(f"Запросов: {len(test_keys)}")
print(f"Linear search: {linear_ms:.3f} ms")
print(f"Hash lookup:   {hash_ms:.3f} ms")

if hash_ms > 0:
    print(f"Отношение времени: {linear_ms / hash_ms:.2f}x")

Запросов: 30
Linear search: 163.098 ms
Hash lookup:   0.171 ms
Отношение времени: 955.19x


## Стоимость построения индекса

In [48]:
start = perf_counter()
tmp_index = build_hash_index(keys)
build_ms = (perf_counter() - start) * 1000

print(f"Build hash index: {build_ms:.3f} ms")

Build hash index: 502.971 ms


---
# 9. Итог сегодняшней версии проекта

Заполните:

**Датасет:**  
https://excelbianalytics.com/downloads-18-sample-csv-files-data-sets-for-testing-sales/$0

**Размер:**  
100000

**Уникальное поле:**  
Order_ID

**Что хранит индекс:**  

Order_ID -> Order_ID

**Поиск без индекса:**  
Линейным поиском

**Сложность поиска без индекса:**  
O(N)

**Average lookup в Hash Table:**  
O(1)

**Worst-case lookup:**  
O(N)

**Почему worst-case может ухудшиться:**  
Из-за коллизий, когда множество ключей попадает в одну корзину

**Начальная capacity:**  
10

**Итоговая capacity после загрузки датасета:**  
163840

**Итоговый load factor:**  
0.6104

**Когда происходит resize:**  
Когда коэффициент заполнения превышает MAX_LOAD_FACTOR = 0.75. При этом capacity увеличивается в два раза.

**Почему при resize нужен rehash:**  
После изменения capacity меняется результат вычисления hash(key) % capacity. Поэтому необходимо заново распределить все элементы по корзинам, чтобы поиск продолжал работать как надо.

**Сколько заняло построение индекса:**  
502.971 ms

**Сколько занял linear search:**  
163.098 ms

**Сколько занял hash lookup:**  
0.171 ms

**Какую цену мы заплатили за быстрый поиск:**  
Дополнительная память O(n) под структуру хэш-таблицы (массив бакетов и списки для разрешения коллизий) и время на первоначальное построение индекса.


---
# 10. Что будет дальше

Сегодня проект умеет только одно:

```text
быстро найти запись по уникальному ключу
```

На следующих занятиях будут появляться другие типы задач и новые структуры

Мы будем **добавлять их только тогда, когда появится конкретная проблема, которую текущая архитектура решает плохо**


---
# Дополнительно, если все основное готово

Добавьте метод:

```python
stats()
```

который возвращает или печатает:

```text
size
capacity
load_factor
количество пустых buckets
размер самого длинного bucket
```

После этого посмотрите статистику для индекса вашего датасета и коротко ответьте:

> Насколько равномерно элементы распределились по buckets?
